# Week 4 · RAG 평가 및 개선

## 목표
- **베이스라인**: Week 3 RAG 그대로 정량 지표 측정
- **개선**: 하이브리드 검색(BM25 + FAISS + RRF) + CrossEncoder Reranker 적용
- **비교**: 전/후 지표 표 + 분석

## 측정 지표
| 지표 | 설명 |
|------|------|
| Recall@K | 상위 K 결과 안에서 relevant 문서를 몇 개 찾았는지 (커버리지) |
| Precision@K | 상위 K 중 실제 relevant 문서 비율 |
| MRR | 첫 번째 relevant 문서의 역수 순위 |
| Hit@K | 상위 K 안에 relevant 문서가 하나라도 있으면 1 |

## 실패 모드 타겟
1. **키워드 불일치**: '양조통'으로 물어볼 때 '술통'이 있는 청크를 못 찾는 문제  
   → BM25(희소) + FAISS(밀집) RRF 혼합으로 양쪽 신호 모두 활용
2. **무관 청크 혼입**: dense retriever가 의미적으로 가깝지만 실제 답이 없는 청크를 상위로 올리는 문제  
   → Cross-Encoder Reranker로 (query, chunk) 쌍 점수를 다시 매겨 상위 4개로 압축

## 0. 환경 설정

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# week4/ 를 작업 디렉토리로 설정
WEEK4_DIR = Path(".").resolve()
if WEEK4_DIR.name != "week4":
    WEEK4_DIR = Path("__file__").parent.resolve()
os.chdir(WEEK4_DIR)
if str(WEEK4_DIR) not in sys.path:
    sys.path.insert(0, str(WEEK4_DIR))

load_dotenv()
print(f"작업 디렉토리: {WEEK4_DIR}")
print(f"OPENAI_API_KEY 설정: {'OK' if os.environ.get('OPENAI_API_KEY') else 'MISSING'}")

작업 디렉토리: /Users/seohyemin/Desktop/langgraph/rag-agent-study/assignments/Parkhaeil/week4
OPENAI_API_KEY 설정: OK


## 1. 베이스라인 RAG 구축 (Week 3 그대로)

In [2]:
from rag.wiki import WikiRetrievalChain
from rag.utils import format_docs

DATA_DIR = "data/docs_sample"

# Week 3 와 동일한 설정 (recursive, chunk_size=800)
baseline_chain = WikiRetrievalChain(
    source_uri=DATA_DIR,
    splitter_type="recursive",
    chunk_size=800,
    chunk_overlap=150,
).create_chain()

print(f"\n총 청크 수: {len(baseline_chain.split_docs)}")
print("\n[청크 ID 샘플]")
for doc in baseline_chain.split_docs[:5]:
    print(f"  {doc.metadata['chunk_id']}  |  {doc.page_content[:50].strip()!r}")

Loaded: crafting_overview.md (제작)
Loaded: farming_ancient_seed.md (농사)
Loaded: farming_crops.md (농사)
Loaded: farming_crops_fall.md (농사)
Loaded: farming_crops_spring.md (농사)
Loaded: farming_crops_summer.md (농사)
Loaded: farming_crops_winter.md (농사)
Loaded: farming_greenhouse.md (농사)
Loaded: farming_keg.md (농사)
Loaded: farming_lightning_rod.md (농사)
Loaded: farming_preserves_jar.md (농사)
Loaded: fishing_fish_list.md (낚시)
Loaded: fishing_overview.md (낚시)
Loaded: fishing_tackle.md (낚시)
Loaded: foraging_overview.md (채집)
Loaded: livestock_animals.md (축산)
Loaded: livestock_barn.md (축산)
Loaded: livestock_coop.md (축산)
Loaded: livestock_mayonnaise.md (축산)
Loaded: mining_dwarf_scroll1.md (광산)
Loaded: mining_dwarf_scroll2.md (광산)
Loaded: mining_overview.md (광산)
Loaded: villagers_hearts.md (주민)
Loaded: villagers_marriage.md (주민)

Loaded 24 documents, 0 failed.
Split into 1655 chunks (recursive, size=800)
FAISS index saved to cache

총 청크 수: 1655

[청크 ID 샘플]
  crafting_overview_0  |  '# Crafting Overvie

## 2. 정답셋(Golden Set) 로드

In [3]:
from eval.golden_set import GOLDEN_SET
from collections import Counter

type_counts = Counter(qa.question_type for qa in GOLDEN_SET)
print(f"총 QA 쌍: {len(GOLDEN_SET)}")
for t, n in type_counts.items():
    print(f"  {t}: {n}개")

print("\n[샘플 — factual]")
for qa in [q for q in GOLDEN_SET if q.question_type == 'factual'][:3]:
    print(f"  Q: {qa.question}")
    print(f"     relevant: {qa.relevant_chunk_ids}")
    print(f"     A: {qa.reference_answer[:70]}...\n")

총 QA 쌍: 35
  factual: 23개
  multi_hop: 6개
  out_of_domain: 6개

[샘플 — factual]
  Q: 술통(양조 설비)을 제작하려면 농사 레벨이 얼마나 필요한가?
     relevant: ['farming_keg']
     A: 농사 레벨 8이 필요합니다....

  Q: 술통 제작에 필요한 재료 4가지는 무엇인가?
     relevant: ['farming_keg']
     A: 나무 30개, 구리 주괴 1개, 철 주괴 1개, 참나무 수지 1개가 필요합니다....

  Q: 과일을 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?
     relevant: ['farming_keg']
     A: 과일은 술통에서 와인으로 가공되며, 판매가는 기본 과일 가격의 3배입니다....



## 3. 베이스라인 검색 지표 측정

In [4]:
from eval.metrics import evaluate_retrieval, aggregate_metrics, print_metrics_table

K = 6

baseline_results = evaluate_retrieval(baseline_chain.retriever, GOLDEN_SET, k=K)
baseline_agg = aggregate_metrics(baseline_results)
print_metrics_table(f"베이스라인 (Recursive RAG, k={K})", baseline_agg)

# 질문별 상세
print("\n[질문별 상세]")
print(f"{'질문':<45} {'type':<12} {'recall':>8} {'mrr':>8} {'hit':>6}")
print("-" * 85)
for r in baseline_results:
    q = r['question'][:43]
    print(f"{q:<45} {r['question_type']:<12} {r[f'recall@{K}']:>8.3f} {r['mrr']:>8.3f} {r[f'hit@{K}']:>6.0f}")


────────────────────────────────────────
  베이스라인 (Recursive RAG, k=6)
────────────────────────────────────────
  recall@6         0.9483
  precision@6      0.5057
  mrr              0.8615
  hit@6            1.0000

[질문별 상세]
질문                                            type           recall      mrr    hit
-------------------------------------------------------------------------------------
술통(양조 설비)을 제작하려면 농사 레벨이 얼마나 필요한가?             factual         1.000    0.500      1
술통 제작에 필요한 재료 4가지는 무엇인가?                      factual         1.000    1.000      1
과일을 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?              factual         1.000    0.250      1
야채를 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?              factual         1.000    1.000      1
밀을 술통에 넣으면 무엇이 만들어지고 소요시간은 얼마인가?              factual         1.000    0.333      1
커피콩 5개를 술통에 넣으면 무엇이 되고 시간은 얼마나 걸리나?           factual         1.000    0.200      1
절임통을 제작하려면 농사 레벨이 얼마나 필요한가?                   factual         1.000    1.000      1
절임통에 과일을 넣으면 어떤 

## 4. 베이스라인 생성 품질 평가 (LLM-as-Judge)

> factual 질문 10개 + out_of_domain 3개를 샘플링하여 채점합니다.

In [5]:
from eval.judge import get_judge_llm, judge_answer, aggregate_judge_results

judge_llm = get_judge_llm(model_name="gpt-4o-mini")

# 샘플: factual 10개 + out_of_domain 3개
factual_sample = [q for q in GOLDEN_SET if q.question_type == 'factual'][:10]
ood_sample = [q for q in GOLDEN_SET if q.question_type == 'out_of_domain'][:3]
eval_sample = factual_sample + ood_sample

print(f"채점 대상: {len(eval_sample)}개 질문")
print("[RAG 답변 생성 + 채점 시작...]")

채점 대상: 13개 질문
[RAG 답변 생성 + 채점 시작...]


/Users/seohyemin/Desktop/langgraph/rag-agent-study/.venv/lib/python3.12/site-packages/langchain/chat_models/base.py:479: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


In [6]:
baseline_judge_results = []

for qa in eval_sample:
    # 검색
    retrieved_docs = baseline_chain.retriever.invoke(qa.question)
    context_str = format_docs(retrieved_docs)
    
    # 생성
    answer = baseline_chain.chain.invoke(
        {"question": qa.question, "context": context_str}
    )
    
    # 채점
    result = judge_answer(
        question=qa.question,
        reference=qa.reference_answer,
        context=context_str,
        answer=answer,
        llm=judge_llm,
    )
    baseline_judge_results.append(result)
    
    c = '✓' if result.correctness.verdict == 'yes' else '✗'
    g = '✓' if result.groundedness.verdict == 'yes' else '✗'
    r_ = '✓' if result.relevance.verdict == 'yes' else '✗'
    print(f"[C:{c} G:{g} R:{r_}] {qa.question[:55]}")

baseline_judge_agg = aggregate_judge_results(baseline_judge_results)
print("\n=== 베이스라인 Judge 집계 ===")
for k, v in baseline_judge_agg.items():
    print(f"  {k:<15} {v:.3f}")

[C:✗ G:✗ R:✗] 술통(양조 설비)을 제작하려면 농사 레벨이 얼마나 필요한가?
[C:✗ G:✗ R:✗] 술통 제작에 필요한 재료 4가지는 무엇인가?
[C:✓ G:✓ R:✓] 과일을 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?
[C:✓ G:✓ R:✓] 야채를 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?
[C:✗ G:✗ R:✓] 밀을 술통에 넣으면 무엇이 만들어지고 소요시간은 얼마인가?
[C:✓ G:✓ R:✓] 커피콩 5개를 술통에 넣으면 무엇이 되고 시간은 얼마나 걸리나?
[C:✓ G:✓ R:✓] 절임통을 제작하려면 농사 레벨이 얼마나 필요한가?
[C:✓ G:✓ R:✓] 절임통에 과일을 넣으면 어떤 제품이 되고, 판매가 계산 공식은?
[C:✗ G:✓ R:✓] 결혼을 제안하려면 상대와의 우정 하트가 몇 개 필요하고, 어떤 아이템이 필요한가?
[C:✓ G:✓ R:✓] 스타듀밸리에서 NPC와 결혼은 싱글플레이에서만 가능한가?
[C:✓ G:✓ R:✓] 스타듀밸리에서 배낭(인벤토리) 크기를 늘리는 방법은?
[C:✓ G:✓ R:✓] 캐릭터 외모(헤어, 피부색 등)를 게임 중에 변경하려면 어떻게 해야 하나?
[C:✓ G:✓ R:✓] 커뮤니티 센터를 완성하면 줄거스마트(Joja) 회사는 어떻게 되나?

=== 베이스라인 Judge 집계 ===
  correctness     0.692
  groundedness    0.769
  relevance       0.846


## 5. 개선된 RAG 구축

### 방법론
1. **하이브리드 검색(RRF)** — BM25(키워드) + FAISS(의미) 결합  
   키워드 불일치 실패 모드 해소
2. **Cross-Encoder Reranker** — 후보 18개(k×3)를 (query, chunk) 쌍 점수로 재정렬 → 상위 4개만 생성에 사용  
   무관 청크 혼입 실패 모드 해소 + 생성 컨텍스트 노이즈 감소

In [7]:
from rag.hybrid import HybridRetriever
from rag.reranker import CrossEncoderReranker

# 하이브리드 검색기 (BM25 + FAISS + RRF)
hybrid_retriever = HybridRetriever(
    vectorstore=baseline_chain.vectorstore,   # FAISS 인덱스 재사용
    split_docs=baseline_chain.split_docs,     # BM25 인덱스 구축
    k=K,
    rrf_k=60,
    fetch_k_multiplier=3,
)

# Cross-Encoder Reranker
# 한국어 성능 향상 시: model_name="BAAI/bge-reranker-v2-m3"
reranker = CrossEncoderReranker(
    model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k=4,
)

print("하이브리드 검색기 & 리랭커 초기화 완료")

# 동작 확인
test_query = "술통 제작에 필요한 재료는?"
hybrid_docs = hybrid_retriever.invoke(test_query)
reranked_docs = reranker.rerank(test_query, hybrid_docs)
print(f"\n[테스트] '{test_query}'")
print(f"  Hybrid 후보: {len(hybrid_docs)}개 → Rerank 후: {len(reranked_docs)}개")
for doc in reranked_docs:
    print(f"  - {doc.metadata.get('chunk_id', '?')}: {doc.page_content[:60].strip()!r}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

하이브리드 검색기 & 리랭커 초기화 완료

[테스트] '술통 제작에 필요한 재료는?'
  Hybrid 후보: 6개 → Rerank 후: 4개
  - farming_keg_18: '### 농장 건물 실내\n\n술통은 농장 건물 안에 설치할 수도 있습니다. [헛간](/%ED%97%9B%EA%B'
  - farming_keg_11: '## 절임통 vs 술통\n\n[절임통](/%EC%A0%88%EC%9E%84%ED%86%B5 "절임통")은 다음과'
  - farming_preserves_jar_11: '## 절임통 vs 술통\n\n:   *참고 문서: [절임통 생산성](/%EC%A0%88%EC%9E%84%ED%8'
  - farming_keg_26: '| [정제 장비](/%EC%A0%9C%EC%9E%91#.EC.A0.95.EC.A0.9C_.EC.9E.A5.E'


## 6. 개선된 검색 지표 측정

### 6-A. 하이브리드(RRF)만 적용

In [8]:
hybrid_results = evaluate_retrieval(hybrid_retriever, GOLDEN_SET, k=K)
hybrid_agg = aggregate_metrics(hybrid_results)
print_metrics_table(f"하이브리드 RAG (BM25+FAISS+RRF, k={K})", hybrid_agg)


────────────────────────────────────────
  하이브리드 RAG (BM25+FAISS+RRF, k=6)
────────────────────────────────────────
  recall@6         0.9310
  precision@6      0.4655
  mrr              0.7741
  hit@6            0.9655


### 6-B. 하이브리드 + Reranker

In [ ]:
class HybridRerankerRetriever:
    """HybridRetriever + CrossEncoderReranker 를 하나의 retriever 인터페이스로 래핑."""
    def __init__(self, hybrid, reranker):
        self.hybrid = hybrid
        self.reranker = reranker

    def invoke(self, query: str):
        candidates = self.hybrid.invoke(query)
        return self.reranker.rerank(query, candidates)


hybrid_reranker = HybridRerankerRetriever(hybrid_retriever, reranker)

# RERANKER_K=4 (실제 반환 문서 수). 검색 지표는 k=K=6 으로 통일해서 비교
# → retrieved[:6] = retrieved[:4] 이므로 recall/hit 값은 @4 와 동일
# precision 분모만 6이 돼서 실질 반환 수(4) 대비 소폭 보수적으로 나오지만 공정 비교 가능
RERANKER_K = reranker.top_k
hr_results = evaluate_retrieval(hybrid_reranker, GOLDEN_SET, k=K)
hr_agg = aggregate_metrics(hr_results)
print_metrics_table(f"하이브리드+Reranker (반환={RERANKER_K}개, 지표 @{K})", hr_agg)

## 7. 개선된 생성 품질 평가 (LLM-as-Judge)

In [10]:
improved_judge_results = []

for qa in eval_sample:
    # 하이브리드+리랭커로 검색
    retrieved_docs = hybrid_reranker.invoke(qa.question)
    context_str = format_docs(retrieved_docs)

    # 생성 (baseline_chain 의 체인 재사용, 검색기만 교체)
    answer = baseline_chain.chain.invoke(
        {"question": qa.question, "context": context_str}
    )

    # 채점
    result = judge_answer(
        question=qa.question,
        reference=qa.reference_answer,
        context=context_str,
        answer=answer,
        llm=judge_llm,
    )
    improved_judge_results.append(result)

    c = '✓' if result.correctness.verdict == 'yes' else '✗'
    g = '✓' if result.groundedness.verdict == 'yes' else '✗'
    r_ = '✓' if result.relevance.verdict == 'yes' else '✗'
    print(f"[C:{c} G:{g} R:{r_}] {qa.question[:55]}")

improved_judge_agg = aggregate_judge_results(improved_judge_results)
print("\n=== 개선 RAG Judge 집계 ===")
for k, v in improved_judge_agg.items():
    print(f"  {k:<15} {v:.3f}")

[C:✗ G:✗ R:✗] 술통(양조 설비)을 제작하려면 농사 레벨이 얼마나 필요한가?
[C:✗ G:✓ R:✗] 술통 제작에 필요한 재료 4가지는 무엇인가?
[C:✓ G:✓ R:✓] 과일을 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?
[C:✓ G:✓ R:✓] 야채를 술통에 넣으면 판매가가 기본 가격의 몇 배가 되나?
[C:✗ G:✗ R:✓] 밀을 술통에 넣으면 무엇이 만들어지고 소요시간은 얼마인가?
[C:✗ G:✗ R:✓] 커피콩 5개를 술통에 넣으면 무엇이 되고 시간은 얼마나 걸리나?
[C:✗ G:✗ R:✗] 절임통을 제작하려면 농사 레벨이 얼마나 필요한가?
[C:✗ G:✗ R:✓] 절임통에 과일을 넣으면 어떤 제품이 되고, 판매가 계산 공식은?
[C:✓ G:✓ R:✓] 결혼을 제안하려면 상대와의 우정 하트가 몇 개 필요하고, 어떤 아이템이 필요한가?
[C:✓ G:✗ R:✓] 스타듀밸리에서 NPC와 결혼은 싱글플레이에서만 가능한가?
[C:✓ G:✓ R:✓] 스타듀밸리에서 배낭(인벤토리) 크기를 늘리는 방법은?
[C:✓ G:✓ R:✓] 캐릭터 외모(헤어, 피부색 등)를 게임 중에 변경하려면 어떻게 해야 하나?
[C:✓ G:✓ R:✓] 커뮤니티 센터를 완성하면 줄거스마트(Joja) 회사는 어떻게 되나?

=== 개선 RAG Judge 집계 ===
  correctness     0.538
  groundedness    0.538
  relevance       0.769


## 8. 전/후 비교 결과 표

In [ ]:
import pandas as pd

# ─── 검색 지표 비교 (모두 k=6 기준) ────────────────
retrieval_comparison = pd.DataFrame([
    {"방법론": f"베이스라인 (Recursive, k={K})", **baseline_agg},
    {"방법론": f"하이브리드 RRF (k={K})", **hybrid_agg},
    {"방법론": f"하이브리드 + Reranker (반환={RERANKER_K}개, @{K})", **hr_agg},
]).set_index("방법론")

print("\n=== 검색 지표 비교 (k=6 기준) ===")
print(retrieval_comparison.to_string(float_format="{:.4f}".format))

# ─── 생성 품질 비교 ─────────────────────────────────
judge_comparison = pd.DataFrame([
    {"방법론": "베이스라인 RAG", **baseline_judge_agg},
    {"방법론": "하이브리드 + Reranker", **improved_judge_agg},
]).set_index("방법론")

print("\n=== LLM-as-Judge 생성 품질 비교 ===")
print(judge_comparison.to_string(float_format="{:.3f}".format))

## 9. 분석

### 실험 결과 요약

| 지표 | 베이스라인 | 하이브리드 RRF | 하이브리드+Reranker |
|------|-----------|--------------|-------------------|
| Recall@6 | **0.9483** | 0.9310 | 0.9310 |
| Precision@6 | **0.5057** | 0.4655 | 0.3793 |
| MRR | **0.8615** | 0.7741 | 0.7098 |
| Hit@6 | **1.0000** | 0.9655 | 0.9655 |
| Correctness (Judge) | **0.692** | — | 0.538 |
| Groundedness (Judge) | **0.769** | — | 0.538 |

### 핵심 발견: 예상과 달랐던 점

**1. 검색 병목이 아니었다**  
베이스라인이 이미 Hit@6=1.0(완벽)이었다. 즉 관련 청크는 top-6 안에 반드시 들어왔다.  
키워드 불일치나 dense 검색 한계로 문서 자체를 못 찾는 문제가 아니라, **찾아왔지만 생성이 틀리는** 문제였다.

**2. 영어 Cross-Encoder가 한국어에서 역효과**  
`cross-encoder/ms-marco-MiniLM-L-6-v2`는 영어 MS-MARCO로 학습된 모델이다. 한국어 청크에 대해 (query, chunk) 점수가 의미 없이 산출되면서 실제로 관련 있는 청크를 아래로 내려보냈다. 테스트 결과에서도 "술통 제작 재료" 쿼리에 대해 제작법 청크(farming_keg_0)가 아닌 건물 배치 청크(farming_keg_18)가 1위로 올라왔다.  
→ **한국어에는 `BAAI/bge-reranker-v2-m3` (다국어 모델)를 사용해야 한다.**

**3. 실제 생성 실패 원인: 표 형식 데이터 파싱 손실**  
"술통 제작 레벨" 질문이 hit@6=1.0임에도 불구하고 C:✗ G:✗를 받은 이유:  
스타듀밸리 위키의 제작법 정보는 이미지(아이콘)가 포함된 테이블 셀에 들어 있다  
(`Farming Skill Icon.png 농사 (8 레벨)`). Markdown 파서가 이미지 태그를 제거하면서 레벨 정보가 맥락 없이 남거나 누락되어 LLM이 답변을 생성하지 못했다.

### 다음 단계 개선 방향

| 우선순위 | 방법 | 기대 효과 |
|----------|------|---------|
| 1 | `BAAI/bge-reranker-v2-m3` 교체 | 한국어 Rerank 품질 회복 |
| 2 | 표 파싱 개선 (pandas/html parser) | 제작법/수치 정보 손실 방지 |
| 3 | Chunk size 축소 (800→400) | 제작법 테이블이 별도 청크로 분리 → MRR 향상 |

In [ ]:
# ─── 생성 실패 질문 상세 분석 ────────────────────────
# 검색(hit@6)은 완벽하지만 생성(judge correctness)이 실패한 질문을 찾는다

print("=== 베이스라인: hit@6=1 이지만 correctness=✗ 질문 (검색 OK, 생성 실패) ===")
hit_key = f"hit@{K}"
hit_by_q = {r['question']: r[hit_key] for r in baseline_results}

gen_fail_baseline = []
for i, qa in enumerate(eval_sample):
    result = baseline_judge_results[i]
    if result.correctness.verdict == 'no':
        hit = hit_by_q.get(qa.question, -1)
        gen_fail_baseline.append((qa.question, hit))
        marker = "검색OK·생성✗" if hit == 1 else "검색✗·생성✗"
        print(f"  [{marker}] {qa.question}")

print(f"\n=== 개선 후(하이브리드+Reranker) correctness 변화 ===")
print(f"{'질문':<48} {'베이스라인':>10} {'개선 후':>8}")
print("-" * 70)
for i, qa in enumerate(eval_sample):
    b = '✓' if baseline_judge_results[i].correctness.verdict == 'yes' else '✗'
    a = '✓' if improved_judge_results[i].correctness.verdict == 'yes' else '✗'
    if b != a:
        change = "⬆ 회복" if a == '✓' else "⬇ 악화"
        print(f"  {qa.question[:46]:<48} {b:>10} {a:>5}  {change}")

# 영어 reranker 가 한국어에서 오작동한 증거: MRR 하락
print(f"\n=== MRR 하락 — 영어 Reranker 오작동 증거 ===")
hr_by_q = {r['question']: r['mrr'] for r in hr_results}
bl_by_q = {r['question']: r['mrr'] for r in baseline_results}
for r in baseline_results:
    q = r['question']
    if q in hr_by_q and bl_by_q[q] > hr_by_q[q]:
        print(f"  MRR {bl_by_q[q]:.2f}→{hr_by_q[q]:.2f}  {q[:55]}")